# Practical 1
### Author: Lan Stare, s1169977

Our first step is to open our files and extract the input information

In [1]:
import sys

""" lines = sys.stdin.read().strip().splitlines() 
for line in lines:
    print(line) """ #This is for the future grading

#This function will be deleted in the future for generalization. Now it's here so I can actively see what my functions are doing
def read_dat(file): #This function traverses the example files and produces a new list where each element (a list of integers) represents the first vertex, the second vertex and the type of edge it is with an exception of the first element, which is constructed of |V| and |E|.
    new_list = []
    with open(f"samples-practice/{file}", "r") as dat:
        dat.readline() #This is to skip the |V| and |E|
        for line in dat:
            new_list.append(list(map(int, line.split())))
    return new_list

file_2 = "2.in"
a = read_dat(file_2)
a

[[1505, 2337, 2],
 [1531, 1980, 1],
 [1209, 1566, 0],
 [420, 1784, 0],
 [982, 1923, 1],
 [51, 476, 1],
 [1603, 1711, 0],
 [1135, 2264, 1],
 [215, 2185, 0],
 [278, 2329, 1],
 [1221, 1477, 1],
 [1735, 1817, 0],
 [1694, 1831, 1],
 [1260, 2202, 0],
 [1354, 1969, 1],
 [925, 2336, 0],
 [1693, 2081, 2],
 [347, 1433, 0],
 [197, 723, 0],
 [876, 1852, 0],
 [1283, 1515, 1],
 [239, 2165, 1],
 [215, 1506, 1],
 [561, 787, 0],
 [468, 866, 0],
 [123, 2328, 1],
 [1119, 1563, 1],
 [232, 862, 0],
 [1216, 1608, 1],
 [766, 2336, 1],
 [555, 2259, 0],
 [1686, 2014, 1],
 [572, 2182, 2],
 [117, 355, 1],
 [2062, 2101, 2],
 [52, 1738, 0],
 [16, 596, 1],
 [147, 1196, 0],
 [1261, 2132, 1],
 [320, 1146, 0],
 [528, 1494, 0],
 [335, 1841, 0],
 [209, 475, 1],
 [175, 2050, 1],
 [1580, 1650, 0],
 [179, 1739, 1],
 [977, 1578, 0],
 [957, 1547, 2],
 [638, 1192, 1],
 [1898, 2462, 0],
 [17, 1294, 1],
 [677, 971, 1],
 [999, 1393, 1],
 [275, 2471, 1],
 [507, 2176, 1],
 [1355, 1789, 1],
 [1138, 2326, 0],
 [1548, 1707, 1],
 [473

Now that we have the inputs neatly in a list of lists, we first need to transform it into two separate lists; one for the busses and one for pedestrians. This is where we assign the weight of the combined roads to be 1 and the weight of all the other roads (either bus or pedestrian only) to be 2. This is done so we can easily run Kruskal's algoristhm to produce the minimum spanning tree. There are different ways we can represent these type of graphs, but since we'll opt for Kruskal's algorithm, using a list of tuples makes the most sense. This is because we don't need to keep track of neighbours which adds unnecesairy time and also we can have duplicates which eliminates the possibility of dictionaries.

In [2]:
def transform_list_into_dict(list):
    pedestrian_dict = []
    bus_dict = [] #Initiating the new lists
    number_of_combined_edges = 0
    
    for v1, v2, type in list:
        if type == 0:
            pedestrian_dict.append((v1, v2, 2)) #setdefault is used to initiate the nested dictionary if it doesn't yet exist inside the v1 values.
        elif type == 1:
            bus_dict.append((v1, v2, 2)) 
        elif type == 2:
            pedestrian_dict.append((v1, v2, 1))
            bus_dict.append((v1, v2, 1))
            number_of_combined_edges += 1
        else:
            raise ValueError("Unexpected type value") #this would happen if the input data is wrong
    
    return (number_of_combined_edges, bus_dict, pedestrian_dict)
                
b = transform_list_into_dict(a)

Now that we have successfully transformed the data into 2 seperate lists, we can actually start with solving our problem.

In [ ]:


def find_min_spanning_tree(graph, mandatory_edges=[]):
        #getting vertices:
    vertices = set()
    for element in graph:
        vertices.add(element[0])
        vertices.add(element[1])
    
        #Initialization of Union-find
    parent = {v: v for v in vertices}
    size = {v: 0 for v in vertices}
    
        #this operation of Union-find finds the name of the set that contains x (root)
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    
        #let's construct union function: it merges two sets into a single set if they are not already connected
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra == rb:
            return False #if they are already in the same set, we would get a cycle by adding it so we return false
        if size[ra] < size[rb]:
            parent[ra] = rb
        elif size[ra] > size[rb]:
            parent[rb] = ra
        else:
            parent[rb] = ra
            size[ra] += 1
        return True
    
        #in this step we want to make use of our mandatory edges. These are used to exclude the other combined edges that 
    
        #let's construct Kruskal's algorithm on graph while using the union-fold:
    #INITIALIZATION
    edges_sorted = sorted(graph, key=lambda e: e[2]) #sorting them by weight (so that we can first choose the combined roads) and adding them only if they connect different components to avoid cycles.
    mst = []
    total_weight = 0
    
    used_combined_edges = [] #we want to remember which combined edges we used
    #MAIN LOOP
    for u, v, w in edges_sorted:
        if union(u, v):
            mst.append((u, v, w))
            total_weight += w
            used_combined_edges.append((u, v))
            if len(mst) == len(vertices) - 1: #this is more efficient than keep checking for all possibilities since we know that for a minimum spanning tree, we'll have exactly |V| - 1 edges
                break
    
    return total_weight, used_combined_edges, mst
            
c = find_min_spanning_tree(b[1])
print(c)
    
    

(3927, [(1505, 2337, 1), (1693, 2081, 1), (572, 2182, 1), (2062, 2101, 1), (957, 1547, 1), (629, 1226, 1), (1288, 1645, 1), (951, 1575, 1), (110, 406, 1), (17, 864, 1), (485, 1269, 1), (7, 1562, 1), (497, 1480, 1), (917, 2477, 1), (1022, 2292, 1), (40, 2357, 1), (681, 1510, 1), (21, 714, 1), (1853, 2164, 1), (812, 1677, 1), (1048, 1250, 1), (23, 991, 1), (1732, 1918, 1), (32, 1856, 1), (1126, 2233, 1), (52, 1712, 1), (706, 1769, 1), (2034, 2090, 1), (1097, 1412, 1), (215, 813, 1), (248, 749, 1), (688, 1115, 1), (1621, 2095, 1), (6, 1210, 1), (2229, 2394, 1), (1241, 1345, 1), (243, 1230, 1), (1985, 2176, 1), (1984, 2225, 1), (243, 2219, 1), (627, 1672, 1), (2079, 2160, 1), (100, 759, 1), (1358, 1522, 1), (900, 1285, 1), (1383, 1676, 1), (707, 1310, 1), (383, 1767, 1), (357, 2291, 1), (1349, 2043, 1), (536, 1217, 1), (498, 633, 1), (248, 1254, 1), (714, 736, 1), (1197, 2328, 1), (345, 1742, 1), (202, 360, 1), (1539, 1822, 1), (948, 1623, 1), (177, 538, 1), (921, 1275, 1), (1401, 2362, 1)

We have implemented the basic Kruskal's algorithm. We now focus on our problem. 

In [ ]:
def return_better_spanning_tree(bus_graph, pedestrian_graph):
    return min(find_min_spanning_tree(bus_graph)[0], find_min_spanning_tree(pedestrian_graph)[0])